In [0]:
from pyspark.sql.functions import *

source_path = '/Volumes/financial_transactions_streaming/source/fraud_watchlist/source_data/'
schema = '/Volumes/financial_transactions_streaming/source/fraud_watchlist/schema/'
checkpoint = '/Volumes/financial_transactions_streaming/source/fraud_watchlist/checkPoint/'

In [0]:
dbutils.fs.ls(source_path)

In [0]:
input_stream = (spark.readStream
                .format('cloudFiles')
                .option('cloudFiles.format','json')
                .option('cloudFiles.schemaLocation',schema)
                .option('cloudFiles.inferSchema',True)
                .load(source_path)
                )

In [0]:
transformed_df = (input_stream.select(
                "*",
                col('_metadata.file_path').alias('file_path'),
                current_timestamp().alias('ingestion_timestamp')
                ))

In [0]:
transformed_df\
        .writeStream.format('delta')\
        .outputMode('Append')\
        .option('checkpointLocation',checkpoint)\
        .trigger(availableNow=True)\
        .toTable("financial_transactions_streaming.bronze.fraud_watchlist_batch_test")

In [0]:
%sql
select * from financial_transactions_streaming.bronze.fraud_watchlist_batch_test